#### Error Reporting

Consider the P0 compiler (or any other compiler). In which module (file) of the compiler or run-time system are the following errors reported?

1. Error in statement `x := x @ 1`
2. Error in statement `while x > 0 then x := x - 1`
3. Error in statement `while x > false do x := x - 1`
4. Error in declaration `var x_y: integer`
5. Error in declaration `var x, y, x: integer`
6. Error in program
```algorithm
procedure p(x, y: integer)
    write(x div y)
program q
    p(7, 0)
```

Each error is reported by the first compiler stage that can detect it. The P0 pipeline is `SC` (scanner) → `P0` (parser + type checker) → `ST` (symbol table, called from the parser) → `CG…` (code generator) → run-time system.

1. **`x := x @ 1`** — reported by the **scanner** (`SC.ipynb`). `@` is not in the P0 alphabet, so `getSym` falls through to `mark('illegal character')`.

2. **`while x > 0 then x := x - 1`** — reported by the **parser** (`P0.ipynb`). After parsing the boolean expression, `whileStatement` expects the keyword `do`; seeing `then` it calls `mark("'do' expected")`.

3. **`while x > false do x := x - 1`** — reported by the **parser / type checker** (`P0.ipynb`). The relation `x > false` mixes `integer` and `boolean` operands; the parser's relation handler marks `'bad type'`.

4. **`var x_y: integer`** — reported by the **scanner** (`SC.ipynb`). Identifiers are `letter {letter | digit}`; `x` is read as an IDENT, then the `_` is an illegal character and `mark('illegal character')` fires.

5. **`var x, y, x: integer`** — reported by the **symbol table** (`ST.ipynb`). When `newDecl` is called for the second `x` in the same scope it finds the existing entry and reports `mark('multiple definition of x')`.

6. **`p(7, 0); write(x div y)`** — NOT reported by any compiler module. It is a run-time error: the divisor is `0`, which traps at execution. On WebAssembly, `i32.div_s` raises a trap; on RISC-V, the P0 run-time system (or the OS via a SIGFPE-like signal) reports the division-by-zero.